# Blinded Relationship Classification

This stage classifies the 100 frozen evidence packets without seeing whether a candidate came from the holdings embedding or matched-control search. It also does not load event returns. OpenAI is used as a conservative document-grounded adjudicator; every supported classification must contain an exact quotation from a supplied pre-event passage.

Responses are checkpointed one candidate at a time. The default batch is ten, so kernel interruptions do not erase completed work.

In [ ]:
from pathlib import Path
import importlib,json,sys
import pandas as pd
ROOT=Path.cwd()
if ROOT.name!='geopolitical_exposure':ROOT=ROOT/'geopolitical_exposure'
sys.path.insert(0,str(ROOT))
PUBLIC=ROOT/'retrieval_benchmark/public'
import src.blinded_relationship_extraction as bre
importlib.reload(bre)
INSTRUCTIONS,SCHEMA=bre.INSTRUCTIONS,bre.SCHEMA
load_packets,freeze_contract,run_batch,validate_responses=bre.load_packets,bre.freeze_contract,bre.run_batch,bre.validate_responses
packets=load_packets(PUBLIC/'blinded_evidence_packets.jsonl')
assert len(packets)==100
print('Frozen blinded packets:',len(packets))

## 1. Inspect and freeze the exact prompt

The prompt prohibits outside knowledge, post-event information, company-name inference, returns, and unsupported industry assumptions. `Unsupported` is an intended outcome rather than a failure.

In [ ]:
print(INSTRUCTIONS)
display(pd.DataFrame({'schema_field':list(SCHEMA['properties'])}))

## 2. Freeze the API contract and view cost scale

The exact approximate input-token count is calculated below from the frozen packets (characters divided by four), rather than hard-coded. Actual billing depends on the model and account pricing. The model, prompt, schema, source hashes, `store=False`, and blinding conditions are fixed in the contract.

In [ ]:
contract,status=freeze_contract(PUBLIC,packets,batch_size=10)
display(status)

## 3. Run the next checkpointed batch

Run this cell repeatedly until `remaining` becomes zero. If the environment variable is unavailable, the cell securely requests the API key without displaying it. Do not paste a key into notebook source.

In [ ]:
failures=run_batch(PUBLIC,packets,contract)
failures.to_parquet(PUBLIC/'latest_relationship_batch_failures.parquet',index=False)
display(failures)
contract,status=freeze_contract(PUBLIC,packets,batch_size=10)
display(status)

## 4. Validate response identity, contract, and exact evidence

A supported label passes only when its quotation appears verbatim in the supplied passages and all hashes agree. Machine validation is necessary but not sufficient: accepted labels still require blinded human review.

In [ ]:
audit,review=validate_responses(PUBLIC,packets,contract)
if len(audit):
    display(audit.groupby(['supported_relationship','machine_valid']).size().rename('responses').reset_index())
    display(audit.loc[~audit.machine_valid] if (~audit.machine_valid).any() else audit.head(20))
else: print('No responses yet.')

## 5. Stop before unblinding

When all 100 responses are checkpointed and machine-valid, review `retrieval_benchmark/public/relationship_human_review.csv` while the candidate source remains hidden. Set `human_decision` to `accept`, `reject`, or `ambiguous`, add notes, and verify that the quoted evidence supports the claimed relationship and direction. Do not open `restricted_truth` and do not load event returns. Unblinding and Precision@K estimation belong to the next stage only after this review is complete.